# Rafiq AI Studio — اختبار فيديو فعلي على GPU متاح مجانًا

هذا دفتر **اختبار برمجي تفاعلي** لمسار ComfyUI نفسه المستخدم في الاستوديو. لا يستضيف موقعًا، ولا ينشئ نفقًا عامًا، ولا يحوّل طلبات الفيديو إلى API مدفوع.

1. استخدم حساب Colab المجاني وحدد GPU مجانيًا من **Runtime → Change runtime type** إن كان متاحًا. لا تشترِ وحدات حوسبة ولا تختَر موردًا مدفوعًا. إن لم يتوفر GPU، توقّف.
2. شغّل الخلايا واحدة تلو الأخرى. التنزيلات والأوزان تستهلك عدة GB من مساحة جلسة Colab.
3. أول اختبار: ثانية واحدة بدقة 320×192 و8 خطوات باستخدام Wan 2.1 1.3B. هذه تجربة مسار، وليست جودة الفيلم النهائية.
4. احفظ الناتج والتقرير، ثم اختر **Runtime → Disconnect and delete runtime**.

**الحالة:** أُعد هذا الدفتر ولم يُنفّذ على GPU بعد. توافر GPU المجاني غير مضمون، وقد تنتهي الجلسة أو تظهر تعارضات اعتماديات. لا تُعرض أي نتيجة تجريبية على أنها توليد ناجح.

[سياسة Colab وحدود الموارد](https://research.google.com/colaboratory/faq.html) · [ترخيص Wan 2.1](https://github.com/Wan-Video/Wan2.1)


In [ ]:
import json, os, shutil, subprocess, sys, time
from pathlib import Path

assert Path('/content').is_dir(), 'Open this notebook in Google Colab.'
assert shutil.which('nvidia-smi'), 'No free NVIDIA GPU is attached. Stop here; do not buy compute units.'
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], check=True)
import torch
assert torch.cuda.is_available(), 'CUDA is not available. Stop and check the free runtime.'
print('GPU:', torch.cuda.get_device_name(0))


## تحميل كود المشروع واعتمادياته

هذه الخلية تثبت البرنامج في الجلسة المؤقتة فقط. تحتفظ بمراجعة Git الفعلية في تقرير الاختبار. لا تربط Google Drive ولا تقرأ ملفات حسابك.


In [ ]:
repo = Path('/content/Rafiq-AI-Studio')
source_ref = 'feature/rafiq-studio-mvp'
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', source_ref,
                    'https://github.com/ism87jo/Rafiq-AI-Studio.git', str(repo)], check=True)
source_commit = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
print('Source commit:', source_commit)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(repo / 'requirements.txt'),
                '-r', str(repo / 'rafiq/studio/requirements.txt'), 'huggingface_hub'], check=True)


## تنزيل ثلاثة ملفات نموذج فقط وتسجيل نسخها

تستخدم Wan 2.1 المرخّص Apache 2.0 للتجربة الأولى. يُحل عنوان المراجعة إلى SHA ثابت، وتُسجّل بصمة SHA‑256 لكل ملف؛ لا تُنزّل مكتبة النماذج بأكملها.


In [ ]:
import hashlib
from huggingface_hub import HfApi, hf_hub_download

model_repo = 'Comfy-Org/Wan_2.1_ComfyUI_repackaged'
model_revision = HfApi().model_info(model_repo).sha
wanted = [
    ('diffusion_models', 'wan2.1_t2v_1.3B_fp16.safetensors'),
    ('text_encoders', 'umt5_xxl_fp8_e4m3fn_scaled.safetensors'),
    ('vae', 'wan_2.1_vae.safetensors'),
]
model_records = []
for category, filename in wanted:
    cached = Path(hf_hub_download(model_repo, filename=f'split_files/{category}/{filename}',
                                  revision=model_revision, cache_dir='/content/rafiq-hf-cache'))
    target = repo / 'models' / category / filename
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        target.symlink_to(cached)
    with target.open('rb') as stream:
        digest = hashlib.file_digest(stream, 'sha256').hexdigest()
    model_records.append(dict(repo=model_repo, revision=model_revision, file=f'{category}/{filename}', sha256=digest))
manifest = Path('/content/rafiq-model-manifest.json')
manifest.write_text(json.dumps(dict(source_commit=source_commit, models=model_records), indent=2))
print(manifest.read_text())


## تشغيل محرك خاص داخل الجلسة

يرتبط المحرك بعنوان loopback فقط؛ لا يوجد رابط عام أو واجهة ويب خارج الدفتر. تُجرى التجربة من الخلية التالية عبر الكود.


In [ ]:
import urllib.request

engine_log = Path('/content/rafiq-comfy.log')
if 'engine_process' not in globals() or engine_process.poll() is not None:
    with engine_log.open('wb') as log:
        engine_process = subprocess.Popen(
            [sys.executable, 'main.py', '--offline', '--listen', '127.0.0.1', '--port', '8188',
             '--disable-auto-launch', '--disable-all-custom-nodes', '--lowvram'],
            cwd=repo, stdout=log, stderr=subprocess.STDOUT,
            env=dict(os.environ, HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1', HF_HUB_DISABLE_TELEMETRY='1'))
engine_ready = False
for attempt in range(180):
    if engine_process.poll() is not None:
        break
    try:
        with urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=2) as response:
            stats = json.load(response)
        print(json.dumps(stats, indent=2))
        engine_ready = True
        break
    except (OSError, TimeoutError):
        time.sleep(1)
if not engine_ready:
    print(engine_log.read_text(errors='replace')[-8000:])
    raise RuntimeError('ComfyUI did not start. No generation was submitted.')


## توليد المقطع وفحص الناتج

الضغط على تشغيل هذه الخلية يبدأ توليدًا فعليًا على GPU الجلسة المحددة. لا يستخدم بديلًا وهميًا عند الفشل. بعد نجاح الاختبار، ارفع الخطوات إلى 20 والمدة إلى 5 ثوانٍ لمراجعة الجودة؛ لا تبدأ بفيلم طويل.


In [ ]:
command = [sys.executable, '-m', 'rafiq.studio.benchmark', '--run', '--preset', 'wan21-1.3b',
           '--seconds', '1', '--steps', '8', '--width', '320', '--height', '192',
           '--data-dir', '/content/rafiq-benchmark', '--model-manifest', str(manifest)]
result = subprocess.run(command, cwd=repo, capture_output=True, text=True, timeout=2100)
print(result.stdout)
if result.returncode:
    print(result.stderr[-5000:])
    print(engine_log.read_text(errors='replace')[-8000:])
    raise RuntimeError('The GPU generation did not pass. Save the error for review.')
summary = json.loads(result.stdout)
assert summary['state'] == 'succeeded' and Path(summary['output_path']).is_file()
from IPython.display import Video, display
display(Video(summary['output_path'], embed=True))


## حفظ النتيجة ثم إيقاف المحرك

هذا التنزيل ينقل ملف MP4 وتقرير الإعدادات إلى جهازك. تقرير القياس يضم زمن التنفيذ وبصمات الأوزان؛ ذروة الذاكرة المرصودة تخص بطاقة GPU كلها، وليست عداد ذاكرة PyTorch داخل العملية وحدها.


In [ ]:
from google.colab import files
import zipfile

bundle = Path('/content/rafiq-first-gpu-test.zip')
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as archive:
    archive.write(summary['output_path'], 'first-shot.mp4')
    archive.write(summary['report'], 'benchmark.json')
    archive.write(manifest, 'model-manifest.json')
files.download(str(bundle))


In [ ]:
if 'engine_process' in globals() and engine_process.poll() is None:
    engine_process.terminate()
    engine_process.wait(timeout=20)
print('Engine stopped. Now use Runtime → Disconnect and delete runtime in Colab.')
